In [1]:
import json
import os
import shutil
from concurrent.futures import ProcessPoolExecutor, as_completed
from tempfile import mkdtemp

import pandas as pd
from tqdm import tqdm

from internal.utils.image_helper import *

# -------------------------
# Config
# -------------------------
DATA_ROOT = "../data"
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, "train_data")
TRAIN_MSK_DIR = os.path.join(DATA_ROOT, "train_data")
TEST_IMG_DIR = os.path.join(DATA_ROOT, "test_data")
TEST_MSK_DIR = os.path.join(DATA_ROOT, "test_data")

PATCH_SIZE = 384
N_PATCHES = 16                  # patches per slide

MARGIN = 16                     # bbox margin (optional use later)

MASK_PATCH_FRAC = 0.75          # fraction of patches that should be mask-driven
MIN_MASK_PIXELS_SLIDE = 50      # if slide mask has fewer pixels -> treat as "no mask"
MIN_MASK_IN_PATCH = 30          # mask pixels required in a "mask" patch
MIN_TISSUE_FRAC = 0.07          # tissue fraction required to accept a patch
MIN_PATCH_PER_SLIDE = 4

MIN_CENTER_DIST = PATCH_SIZE // 2
MAX_TRIES_PER_SLIDE = 10      # hard cap to prevent infinite loops
MAX_TRIES_PER_PATCH = 5        # local tries before relaxing constraints


OUT_DIR = os.path.join("processed", "numpy_patches")
os.makedirs(OUT_DIR, exist_ok=True)

# train
TRAIN_X_PATH = os.path.join(OUT_DIR, f"train_x_uint8_{PATCH_SIZE}.dat")
TRAIN_M_PATH = os.path.join(OUT_DIR, f"train_m_uint8_{PATCH_SIZE}.dat")
TRAIN_Y_PATH = os.path.join(OUT_DIR, "train_y_int64.npy")
TRAIN_IDX_PATH = os.path.join(OUT_DIR, "train_slide_to_patchidx.npy")
TRAIN_LOG_PATH = os.path.join(OUT_DIR, "train_build_log.csv")
# test
TEST_X_PATH = os.path.join(OUT_DIR, f"test_x_uint8_{PATCH_SIZE}.dat")
TEST_M_PATH = os.path.join(OUT_DIR, f"test_m_uint8_{PATCH_SIZE}.dat")
# TEST_Y_PATH = os.path.join(OUT_DIR, "test_y_int64.npy")
TEST_IDX_PATH = os.path.join(OUT_DIR, "test_slide_to_patchidx.npy")
TEST_LOG_PATH = os.path.join(OUT_DIR, "test_build_log.csv")

# class mapping
CLASS2ID: dict[str, int] = {"Luminal A": 0, "Luminal B": 1, "HER2(+)": 2, "Triple negative": 3}
ID2CLASS: dict[int, str] = {0:"Luminal A", 1:"Luminal B", 2:"HER2(+)", 3:"Triple negative"}

# assertions
assert os.path.exists(TRAIN_IMG_DIR), f"Train image dir not found: {TRAIN_IMG_DIR}"
assert os.path.exists(TRAIN_MSK_DIR), f"Train mask dir not found: {TRAIN_MSK_DIR}"
for key, val in CLASS2ID.items():
    assert ID2CLASS[val] == key, "CLASS2ID and ID2CLASS mismatch for class " + key

# save configuration used into a dataframe
config = {
    "OUT_DIR": OUT_DIR,
    # train
    "X_PATH": TRAIN_X_PATH,
    "M_PATH": TRAIN_M_PATH,
    "Y_PATH": TRAIN_Y_PATH,
    "IDX_PATH": TRAIN_IDX_PATH,
    "LOG_PATH": TRAIN_LOG_PATH,
    # test
    "TEST_X_PATH": TEST_X_PATH,
    "TEST_M_PATH": TEST_M_PATH,
    # "TEST_Y_PATH": TEST_Y_PATH,
    "TEST_IDX_PATH": TEST_IDX_PATH,
    "TEST_LOG_PATH": TEST_LOG_PATH,
    # -------------------------
    "PATCH_SIZE": PATCH_SIZE,
    "N_PATCHES": N_PATCHES,
    "MARGIN": MARGIN,
    "MASK_PATCH_FRAC": MASK_PATCH_FRAC,
    "MIN_MASK_PIXELS_SLIDE": MIN_MASK_PIXELS_SLIDE,
    "MIN_MASK_IN_PATCH": MIN_MASK_IN_PATCH,
    "MIN_TISSUE_FRAC": MIN_TISSUE_FRAC,
    "MIN_PATCH_PER_SLIDE": MIN_PATCH_PER_SLIDE,
    "MIN_CENTER_DIST": MIN_CENTER_DIST,
    "MAX_TRIES_PER_SLIDE": MAX_TRIES_PER_SLIDE,
    "MAX_TRIES_PER_PATCH": MAX_TRIES_PER_PATCH,
    "CLASS2ID": CLASS2ID,
    "ID2CLASS": ID2CLASS,
}
with open(os.path.join("processed", "config.json"), "w") as f:
    json.dump(config, f, indent=4)

In [2]:
# -------------------------
# DataFrame cleanup
# -------------------------
def cleanup_train_labels(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove entries from df where the corresponding image file does not exist.
    :param df: DataFrame with at least a "sample_index" column
    :return: Cleaned DataFrame
    """
    to_drop = []
    for idx, row in df.iterrows():
        sample_index = row["sample_index"]
        img_path = os.path.join(TRAIN_IMG_DIR, sample_index)
        if not os.path.exists(img_path):
            to_drop.append(idx)
    df = df.drop(to_drop).reset_index(drop=True)
    return df

In [3]:
def _process_one_slide(
    s, slide_id: str, label_id,
    img_dir: str, msk_dir: str | None,
    patch_size, n_patches,   # n_patches is now "max target", not mandatory
    mask_patch_frac, min_mask_pixels_slide, min_mask_in_patch,
    min_tissue_frac, min_dist_tissue, min_dist_mask,
    max_tries_per_slide, max_tries_per_patch,
    base_seed,
    tmp_dir: str,
    min_patches_per_slide: int = 1,
):
    rng = np.random.default_rng(base_seed + 1000003 * s)

    img_path = os.path.join(img_dir, slide_id)
    rgb = read_rgb(img_path)

    # --- mask optional ---
    if msk_dir is not None:
        msk_path = os.path.join(msk_dir, slide_id.replace("img", "mask"))
        msk = read_mask(msk_path).astype(np.uint8)
        if msk.mean() > 0.5:
            msk = 1 - msk
    else:
        # dummy all-zero mask => tissue-only sampling, and M will be zeros
        h, w = rgb.shape[:2]
        msk = np.zeros((h, w), dtype=np.uint8)

    h, w = rgb.shape[:2]
    half = patch_size // 2

    if min_dist_tissue is None:
        min_dist_tissue = patch_size // 2
    if min_dist_mask is None:
        min_dist_mask = patch_size // 4
    min_dist_sq_tissue = int(min_dist_tissue * min_dist_tissue)
    min_dist_sq_mask   = int(min_dist_mask * min_dist_mask)

    tmask = tissue_mask_from_rgb_lab(rgb)
    top_h = detect_top_shadow_by_step(rgb)   # compute once per slide

    mask_pixels_slide = int(msk.sum())
    components = get_mask_components(msk) if mask_pixels_slide >= min_mask_pixels_slide else []

    k_mask_target = int(round(n_patches * mask_patch_frac))
    if len(components) == 0:
        k_mask_target = 0
    else:
        MAX_PATCHES_PER_COMPONENT = 2
        k_mask_target = min(k_mask_target, MAX_PATCHES_PER_COMPONENT * len(components))

    # --- VARIABLE outputs ---
    X_list = []
    M_list = []

    centers = []
    boxes = []
    seen_hashes = []
    MAX_DHASH_DIST = 3

    mask_done = 0
    tissue_done = 0
    produced = 0
    tries = 0

    # NOTE: we still "aim" for n_patches, but we won't pad if we fail
    while produced < n_patches and tries < max_tries_per_slide:
        tries += 1

        use_mask = (mask_done < k_mask_target) and (len(components) > 0)

        if use_mask and tries > (max_tries_per_slide // 5):
            k_mask_target = mask_done
            use_mask = False

        accepted = False
        for _ in range(max_tries_per_patch):
            if use_mask:
                cx, cy = sample_from_components(components, rng)
            else:
                c = sample_from_binary(tmask, rng)
                if c is None:
                    cx = int(rng.integers(half, w - half))
                    cy = int(rng.integers(half, h - half))
                else:
                    cx, cy = c

            cx, cy = clamp_center(cx, cy, w, h, half)

            # ---- avoid centers that would include top shadow band ----
            if top_h > 0 and (cy - half) < top_h:
                continue

            min_dist_sq_use = min_dist_sq_mask if use_mask else min_dist_sq_tissue
            if not far_enough(cx, cy, centers, min_dist_sq_use):
                continue

            new_box = box_from_center(cx, cy, half)
            if too_much_overlap(new_box, boxes, max_iou=0.7):
                continue

            patch_rgb = crop_patch(rgb, cx, cy, patch_size)
            patch_msk = crop_patch(msk, cx, cy, patch_size)

            if has_top_shadow(patch_rgb):
                continue

            if patch_rgb.shape[:2] != (patch_size, patch_size):
                continue

            tmask_patch = crop_patch(tmask, cx, cy, patch_size)
            if tmask_patch.shape[:2] != (patch_size, patch_size):
                continue
            tissue_frac_patch = float(tmask_patch.mean())
            if tissue_frac_patch < min_tissue_frac:
                continue

            gray = patch_rgb.mean(axis=2)
            if gray.std() < 5.0:
                continue

            mask_px = int((patch_msk > 0).sum())
            if use_mask and mask_px < min_mask_in_patch:
                continue

            ph = imagehash.dhash(Image.fromarray(patch_rgb), hash_size=8)
            if any((ph - hsh) <= MAX_DHASH_DIST for hsh in seen_hashes):
                continue
            seen_hashes.append(ph)

            # accept
            centers.append((cx, cy))
            boxes.append(new_box)

            X_list.append(patch_rgb.astype(np.uint8)) # shape (patch_size, patch_size, 3) (H, W, 3)
            M_list.append(((patch_msk > 0).astype(np.uint8) * 255)[:, :, None]) # mask shape (patch_size, atch_size, 1) (H, W, 1)

            produced += 1
            if use_mask:
                mask_done += 1
            else:
                tissue_done += 1

            accepted = True
            break

        if not accepted:
            # optional relax to avoid deadlocks
            if len(centers) > 8:
                centers = centers[-8:]
            if not use_mask:
                min_dist_sq_tissue = max(min_dist_sq_tissue // 2, (patch_size // 8) ** 2)
            else:
                min_dist_sq_mask = max(min_dist_sq_mask // 2, (patch_size // 16) ** 2)

    if produced < min_patches_per_slide:
        raise RuntimeError(f"Could not extract enough patches for slide {slide_id} (produced={produced})")

    # stack variable
    X_block = np.stack(X_list, axis=0)
    M_block = np.stack(M_list, axis=0)

    # write to temp file (avoid returning big arrays through IPC)
    tmp_path: str = os.path.join(tmp_dir, f"slide_{s:06d}.npz")
    np.savez(tmp_path, X=X_block, M=M_block)

    log_row: dict[str, str | int] = {
        "slide_id": slide_id,
        "produced": int(produced),
        "mask_done": int(mask_done),
        "tissue_done": int(tissue_done),
        "mask_pixels_slide": int(mask_pixels_slide),
        "n_components": int(len(components)),
        "tries": int(tries),
        "status": "ok" if produced == n_patches else "partial",
        # patch_range_* will be filled in finalize phase
    }

    return s, tmp_path, log_row


def build_numpy_parallel(
        slide_ids: list[str],
        *,
        img_dir: str,
        msk_dir: str | None,
        out_x_path: str,
        out_m_path: str,
        out_y_path: str | None,
        out_idx_path: str,
        out_log_path: str,
        y: np.ndarray | None = None,              # (n_slides,) or None for test
        patch_size: int = 384,
        n_patches: int = 16,
        mask_patch_frac: float = 0.75,
        min_mask_pixels_slide: int = 50,
        min_mask_in_patch: int = 10,
        min_tissue_frac: float = 0.07,
        min_dist_tissue: int | None = None,
        min_dist_mask: int | None = None,
        max_tries_per_slide: int = 8000,
        max_tries_per_patch: int = 80,
        seed: int = 42,
        n_workers: int | None = None,
        min_patches_per_slide: int = 1,
):
    os.makedirs(os.path.dirname(out_x_path), exist_ok=True)

    n_slides = len(slide_ids)
    if y is None:
        y = np.full((n_slides,), -1, dtype=np.int64)   # dummy labels for test

    if min_dist_tissue is None:
        min_dist_tissue = patch_size // 2
    if min_dist_mask is None:
        min_dist_mask = patch_size // 4

    n_workers = n_workers or max(1, (os.cpu_count() or 2) - 1)
    tmp_dir = mkdtemp(prefix="patch_build_tmp_")

    try:
        results = [None] * n_slides
        logs = []

        with ProcessPoolExecutor(max_workers=n_workers) as ex:
            futures = [
                ex.submit(
                    _process_one_slide,
                    s, slide_id, int(y[s]),
                    img_dir, msk_dir,
                    patch_size, n_patches,
                    mask_patch_frac, min_mask_pixels_slide, min_mask_in_patch,
                    min_tissue_frac, min_dist_tissue, min_dist_mask,
                    max_tries_per_slide, max_tries_per_patch,
                    seed, tmp_dir, min_patches_per_slide,
                )
                for s, slide_id in enumerate(slide_ids)
            ]

            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc="Extracting patches", unit="slide"):
                s, tmp_path, log_row = fut.result()
                with np.load(tmp_path) as z:
                    produced = int(z["X"].shape[0])
                results[s] = (tmp_path, log_row, produced)
                logs.append(log_row)

        produced_counts = np.array([results[s][2] for s in range(n_slides)], dtype=np.int64)
        offsets = np.zeros(n_slides + 1, dtype=np.int64)
        offsets[1:] = np.cumsum(produced_counts)
        total_patches = int(offsets[-1])

        X = np.memmap(out_x_path, dtype=np.uint8, mode="w+",
                      shape=(total_patches, patch_size, patch_size, 3))
        M = np.memmap(out_m_path, dtype=np.uint8, mode="w+",
                      shape=(total_patches, patch_size, patch_size, 1))
        slide_to_patchidx = np.zeros((n_slides, 2), dtype=np.int64)

        for s in tqdm(range(n_slides), desc="Writing memmaps", unit="slide"):
            start = int(offsets[s])
            end = int(offsets[s + 1])

            tmp_path, log_row, _ = results[s]
            with np.load(tmp_path) as z:
                X[start:end] = z["X"]
                M[start:end] = z["M"]

            slide_to_patchidx[s] = (start, end)
            log_row["patch_range_start"] = start
            log_row["patch_range_end"] = end

        np.save(out_idx_path, slide_to_patchidx)
        if out_y_path is not None:
            np.save(out_y_path, y.astype(np.int64))
        pd.DataFrame(logs).to_csv(out_log_path, index=False)

        X.flush(); M.flush()

        return {
            "x_path": out_x_path,
            "m_path": out_m_path,
            "y_path": out_y_path,
            "idx_path": out_idx_path,
            "log_path": out_log_path,
            "n_slides": n_slides,
            "total_patches": total_patches,
        }
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

# build the dataset
train_csv_path = os.path.join(DATA_ROOT, "train_labels.csv")
df = cleanup_train_labels(pd.read_csv(train_csv_path))
# save cleaned df
df.to_csv(os.path.join("processed", "train_labels_cleaned.csv"), index=False)
train_summary = build_numpy_parallel(
    df["sample_index"].tolist(),
    img_dir=TRAIN_IMG_DIR,
    msk_dir=TRAIN_MSK_DIR,
    out_x_path=TRAIN_X_PATH,
    out_m_path=TRAIN_M_PATH,
    out_y_path=TRAIN_Y_PATH,
    out_idx_path=TRAIN_IDX_PATH,
    out_log_path=TRAIN_LOG_PATH,
    y=df["label"].map(CLASS2ID).to_numpy(dtype=np.int64),
    patch_size=PATCH_SIZE,
    n_patches=N_PATCHES,
    mask_patch_frac=MASK_PATCH_FRAC,
    min_mask_pixels_slide=MIN_MASK_PIXELS_SLIDE,
    min_mask_in_patch=MIN_MASK_IN_PATCH,
    min_tissue_frac=MIN_TISSUE_FRAC,
    max_tries_per_slide=MAX_TRIES_PER_SLIDE,
    seed=42,
    min_patches_per_slide=MIN_PATCH_PER_SLIDE,
)
print(f"Train summary: {json.dumps(train_summary, indent=2)}")

# test_df = pd.read_csv(os.path.join(DATA_ROOT, "test_labels.csv"))  # or sample_submission.csv
# since test_labels.csv is not provided, we create a dummy dataframe
test_image_filenames = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith('.png') and f.startswith("img_")]
test_df = pd.DataFrame({"sample_index": test_image_filenames})
test_summary = build_numpy_parallel(
    test_df["sample_index"].tolist(),
    img_dir=TEST_IMG_DIR,
    msk_dir=TEST_MSK_DIR,
    out_x_path=TEST_X_PATH,
    out_m_path=TEST_M_PATH,
    out_y_path=None,                  # <- no labels for test
    out_idx_path=TEST_IDX_PATH,
    out_log_path=TEST_LOG_PATH,
    patch_size=PATCH_SIZE,
    n_patches=N_PATCHES,
    mask_patch_frac=MASK_PATCH_FRAC,
    min_mask_pixels_slide=MIN_MASK_PIXELS_SLIDE,
    min_mask_in_patch=MIN_MASK_IN_PATCH,
    min_tissue_frac=MIN_TISSUE_FRAC,
    max_tries_per_slide=MAX_TRIES_PER_SLIDE,
    seed=42,
    min_patches_per_slide=MIN_PATCH_PER_SLIDE,
)
print(f"Test summary: {json.dumps(test_summary, indent=2)}")

Writing memmaps: 100%|██████████| 581/581 [00:03<00:00, 157.30slide/s]


Train summary: {
  "x_path": "processed/numpy_patches/train_x_uint8_384.dat",
  "m_path": "processed/numpy_patches/train_m_uint8_384.dat",
  "y_path": "processed/numpy_patches/train_y_int64.npy",
  "idx_path": "processed/numpy_patches/train_slide_to_patchidx.npy",
  "log_path": "processed/numpy_patches/train_build_log.csv",
  "n_slides": 581,
  "total_patches": 5122
}


Writing memmaps: 100%|██████████| 477/477 [00:02<00:00, 164.69slide/s]


Test summary: {
  "x_path": "processed/numpy_patches/test_x_uint8_384.dat",
  "m_path": "processed/numpy_patches/test_m_uint8_384.dat",
  "y_path": null,
  "idx_path": "processed/numpy_patches/test_slide_to_patchidx.npy",
  "log_path": "processed/numpy_patches/test_build_log.csv",
  "n_slides": 477,
  "total_patches": 4208
}
